# Transferibilidad a Colombia — 01: Ablacion por Sensor y Robustez ante Nubosidad

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/transferibilidad_01_ablacion_robustez.ipynb)

**Preguntas clave:**
- Si en Colombia el satelite Sentinel-2 esta nublado (60-80% del tiempo), cuanto cae el F1?
- Cual grupo de sensores es mas critico para detectar deslizamientos?
- Cuanto depende el modelo de bandas opticas vs SAR vs DEM?

> Cambia `DRIVE_PATH` antes de correr.

## 0. Configuracion

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado')

In [ ]:
from pathlib import Path
import sys
DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'
ROOT=Path(DRIVE_PATH); IMG_DIR=ROOT/'TrainData'/'img'; MASK_DIR=ROOT/'TrainData'/'mask'
OUT_DIR=ROOT/'results'/'transferibilidad_01'; OUT_DIR.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(ROOT))
assert len(list(IMG_DIR.glob('*.h5')))>0,'No se encontraron imagenes'
print(f'Imagenes: {len(list(IMG_DIR.glob("*.h5")))}  |  Salida: {OUT_DIR}')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import h5py, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, precision_recall_curve, average_precision_score
from tqdm.notebook import tqdm
%matplotlib inline
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})

CHANNEL_NAMES=['S2-B2 Azul','S2-B3 Verde','S2-B4 Rojo','S2-B8 NIR','S2-B8A NIR-A',
               'S2-B11 SWIR1','S2-B12 SWIR2','S1-VV SAR','S1-VH SAR',
               'ALOS DEM','ALOS Pendiente','S2-B5 RE1','S2-B6 RE2','S2-B7 RE3']
CHANNEL_MEAN=[0.1245,0.1438,0.1312,0.2891,0.3015,0.2134,0.1789,0.0823,0.0641,
              0.4521,0.2189,0.3102,0.3478,0.3812]
CHANNEL_STD=[0.0512,0.0621,0.0589,0.0934,0.0978,0.0734,0.0612,0.0341,0.0289,
             0.2134,0.1456,0.0812,0.0867,0.0923]
# Grupos de canales
GRUPOS={'S2 Optico':[0,1,2,3,4,5,6],'SAR (VV/VH)':[7,8],
        'DEM+Pendiente':[9,10],'Red-Edge':[11,12,13],
        'Todo optico S2+RE':[0,1,2,3,4,5,6,11,12,13]}
N_SAMPLES=600; SEED=42
print('Imports OK')

## 1. Carga de datos

In [ ]:
def normalize(patch):
    m=np.array(CHANNEL_MEAN,dtype=np.float32).reshape(1,1,-1)
    s=np.array(CHANNEL_STD, dtype=np.float32).reshape(1,1,-1)
    return (patch-m)/(s+1e-8)

def extract_features(patch):
    eps=1e-8
    nir,rojo,verde,azul=patch[:,:,3],patch[:,:,2],patch[:,:,1],patch[:,:,0]
    vv,vh=patch[:,:,7],patch[:,:,8]
    ndvi=(nir-rojo)/(nir+rojo+eps); ndwi=(verde-nir)/(verde+nir+eps)
    sar_cr=vh/(vv+eps); evi=2.5*(nir-rojo)/(nir+6*rojo-7.5*azul+1+eps)
    return np.concatenate([patch.mean(axis=(0,1)),patch.std(axis=(0,1)),
                           [ndvi.mean(),ndwi.mean(),sar_cr.mean(),evi.mean()]]).astype(np.float32)

def load_dataset(n=N_SAMPLES,seed=SEED):
    img_files=sorted(IMG_DIR.glob('*.h5'))
    rng=np.random.default_rng(seed)
    chosen=sorted(rng.choice(len(img_files),size=min(n,len(img_files)),replace=False))
    patches,X_list,y_list=[],[],[]
    for i in tqdm(chosen,desc='Cargando'):
        img_f=img_files[i]; mask_f=MASK_DIR/img_f.name.replace('image_','mask_')
        if not mask_f.exists(): continue
        with h5py.File(img_f,'r') as f: patch=f['img' if 'img' in f else list(f.keys())[0]][()].astype(np.float32)
        with h5py.File(mask_f,'r') as f: mask=f['mask' if 'mask' in f else list(f.keys())[0]][()]
        patches.append(patch)
        X_list.append(extract_features(normalize(patch)))
        y_list.append(int(mask.max()>0))
    X,y=np.stack(X_list),np.array(y_list)
    print(f'Dataset: {X.shape[0]} muestras | {y.sum()} positivos ({100*y.mean():.1f}%)')
    return np.array(patches),X,y

patches_raw,X,y=load_dataset()
print(f'X shape: {X.shape}')

## 2. Ablacion por grupo de sensores

Para cada grupo de canales, lo eliminamos (ponemos a cero) y medimos la caida de F1.
Esto responde directamente: **que sensor es mas critico para Colombia?**

In [ ]:
def features_masked(patches_raw, mask_channels):
    'Extrae features enmascarando los canales indicados (los pone a 0).'
    X_m=[]
    for patch in patches_raw:
        p=patch.copy(); p[:,:,mask_channels]=0.0
        X_m.append(extract_features(normalize(p)))
    return np.stack(X_m)

def eval_rf(X_eval,y,seed=SEED,cv=5):
    rf=RandomForestClassifier(n_estimators=150,class_weight='balanced',
                              min_samples_leaf=2,n_jobs=-1,random_state=seed)
    scores=cross_val_score(rf,X_eval,y,cv=StratifiedKFold(cv,shuffle=True,random_state=seed),
                           scoring='f1',n_jobs=-1)
    return scores.mean(),scores.std()

print('Evaluando ablaciones (puede tardar ~5 min)...')
resultados={'Linea base (14 ch)': eval_rf(X,y)}
for grupo,canales in GRUPOS.items():
    print(f'  Enmascarando: {grupo} ({len(canales)} canales)...')
    X_m=features_masked(patches_raw,canales)
    resultados[f'Sin {grupo}']=eval_rf(X_m,y)

print('\nResultados:')
for k,(m,s) in resultados.items():
    caida=(resultados['Linea base (14 ch)'][0]-m)*100
    signo='+' if caida<=0 else '-'
    print(f'  {k:<30} F1={m:.4f}+/-{s:.4f}  ({signo}{abs(caida):.2f} pp)')

In [ ]:
# Figura: impacto de ablacion
nombres=list(resultados.keys()); medias=[v[0] for v in resultados.values()]
errs   =[v[1] for v in resultados.values()]
base   =medias[0]
caidas =[base-m for m in medias]

fig,axes=plt.subplots(1,2,figsize=(14,5))
# F1 absoluto
ax=axes[0]
colors=['#2E5FA3' if i==0 else '#D97706' if c>0.02 else '#16A34A'
        for i,(c,_) in enumerate(zip(caidas,errs))]
bars=ax.barh(range(len(nombres)),medias,xerr=errs,color=colors,
             edgecolor='white',linewidth=0.5,capsize=4)
ax.set_yticks(range(len(nombres))); ax.set_yticklabels(nombres,fontsize=9)
ax.axvline(base,color='navy',lw=1.5,ls='--',label=f'Linea base F1={base:.3f}')
ax.set_xlabel('F1-Score (5-fold CV)',fontsize=11)
ax.set_title('F1 segun grupo de canales enmascarado',fontsize=11,fontweight='bold')
ax.legend(fontsize=9); ax.invert_yaxis()
# Caida en pp
ax=axes[1]
caidas_pp=[(base-m)*100 for m in medias[1:]]
y_pos=range(len(nombres)-1)
c2=['#EF4444' if c>2 else '#F59E0B' if c>0.5 else '#16A34A' for c in caidas_pp]
ax.barh(y_pos,caidas_pp,color=c2,edgecolor='white',linewidth=0.5)
ax.set_yticks(y_pos); ax.set_yticklabels(nombres[1:],fontsize=9)
ax.axvline(0,color='gray',lw=0.8); ax.invert_yaxis()
ax.set_xlabel('Caida en F1 (puntos porcentuales)',fontsize=11)
ax.set_title('Impacto de perder cada grupo de sensores\n(rojo = critico, verde = prescindible)',
             fontsize=11,fontweight='bold')
for i,(v,n) in enumerate(zip(caidas_pp,nombres[1:])):
    ax.text(v+0.05,i,f'{v:.2f} pp',va='center',fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR/'ablacion_sensores.png',dpi=150,bbox_inches='tight')
plt.show(); print('Guardado: ablacion_sensores.png')

## 3. Robustez ante nubosidad colombiana

Simulacion de cobertura de nubes: enmascarar progresivamente los canales **opticos** de Sentinel-2
al 30%, 60% y 80% de los pixeles (equivale a nubosidad parcial por parche).
El SAR (VV/VH) NO se enmascara — penetra nubes.

**Hipotesis:** el modelo deberia degradarse menos si el SAR es discriminativo.

In [ ]:
def simulate_cloud(patches_raw, cloud_fraction, optical_chs=list(range(7))+[11,12,13], seed=0):
    'Enmascara cloud_fraction de los pixeles en canales opticos de cada parche.'
    rng=np.random.default_rng(seed)
    X_cloud=[]
    for patch in patches_raw:
        p=patch.copy()
        H,W=p.shape[:2]
        n_masked=int(H*W*cloud_fraction)
        if n_masked>0:
            flat_idx=rng.choice(H*W,size=n_masked,replace=False)
            rows,cols=np.unravel_index(flat_idx,(H,W))
            p[rows,cols,:]=0.0  # enmascarar todos los canales opticos
        X_cloud.append(extract_features(normalize(p)))
    return np.stack(X_cloud)

# Entrenar RF base completo
rf_base=RandomForestClassifier(n_estimators=150,class_weight='balanced',
                               min_samples_leaf=2,n_jobs=-1,random_state=SEED)
rf_base.fit(X,y)
f1_base=f1_score(y,rf_base.predict(X))
print(f'RF entrenado — F1 train (referencia): {f1_base:.4f}')

# Simular distintos niveles de nubosidad
niveles=[0.0,0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90]
f1_cloud,f1_sar_only=[],[]

print('Simulando nubosidad...')
for frac in tqdm(niveles,desc='Niveles'):
    # Con nubosidad (SAR intacto)
    X_c=simulate_cloud(patches_raw,frac)
    f1_cloud.append(f1_score(y,rf_base.predict(X_c)))
    # Solo SAR+DEM (simulacion extrema: 100% nubes opticas)
    X_sar=features_masked(patches_raw,[0,1,2,3,4,5,6,11,12,13])
    f1_sar_only.append(eval_rf(X_sar,y,cv=3)[0])

print('Niveles | F1 con nubes | F1 solo SAR+DEM')
for n,fc,fs in zip(niveles,f1_cloud,f1_sar_only):
    print(f'  {int(n*100):3d}%  |  {fc:.4f}        |  {fs:.4f}')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))

ax=axes[0]
ax.plot([n*100 for n in niveles],f1_cloud,'o-',color='#2E5FA3',lw=2,ms=6,
        label='RF (SAR intacto, optico enmascarado)')
ax.axhline(f1_sar_only[-1],color='#D97706',lw=1.8,ls='--',
           label=f'Solo SAR+DEM (F1={f1_sar_only[-1]:.3f})')
ax.axvspan(60,100,alpha=0.08,color='gray',label='Nubosidad tipica Colombia (60-80%)')
ax.set_xlabel('Cobertura de nubes simulada (%)',fontsize=11)
ax.set_ylabel('F1-Score',fontsize=11)
ax.set_title('Robustez ante nubosidad\n(degradacion del F1 con perdida de banda optica)',
             fontsize=11,fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(0,1)
ax.grid(axis='y',linestyle='--',alpha=0.4)

ax=axes[1]
caida_pp=[(f1_cloud[0]-f)*100 for f in f1_cloud]
ax.bar([n*100 for n in niveles],caida_pp,
       color=['#16A34A' if c<1 else '#F59E0B' if c<5 else '#EF4444' for c in caida_pp],
       width=7,edgecolor='white')
ax.axvspan(60,100,alpha=0.08,color='gray',label='Zona Colombia')
ax.set_xlabel('Cobertura de nubes (%)',fontsize=11)
ax.set_ylabel('Caida en F1 (pp)',fontsize=11)
ax.set_title('Degradacion acumulada por nubosidad',fontsize=11,fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y',linestyle='--',alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR/'robustez_nubosidad.png',dpi=150,bbox_inches='tight')
plt.show(); print('Guardado: robustez_nubosidad.png')

## 4. Mapa de criticidad de sensores para Colombia

In [ ]:
# Resumen integrado: ablacion + nubosidad
fig,ax=plt.subplots(figsize=(10,5))

# Datos
sensores=['SAR\n(VV/VH)','DEM+\nPendiente','Red-Edge\n(B5-B7)','Optico S2\n(B2-B12)']
caida_ablacion=[(base-resultados.get(f'Sin {g}',resultados['Linea base (14 ch)'])[0])*100
                for g in ['SAR (VV/VH)','DEM+Pendiente','Red-Edge','Todo optico S2+RE']]
disponibilidad_colombia=[95,98,80,30]  # % tiempo disponible (SAR siempre, optico solo dias claros)

# Scatter: criticidad vs disponibilidad
sc=ax.scatter(disponibilidad_colombia,caida_ablacion,
              s=[abs(c)*200+100 for c in caida_ablacion],
              c=caida_ablacion,cmap='RdYlGn_r',
              edgecolors='white',linewidths=1.5,zorder=3)
for i,(s,x,y_) in enumerate(zip(sensores,disponibilidad_colombia,caida_ablacion)):
    ax.annotate(s,(x,y_),textcoords='offset points',xytext=(8,8),fontsize=9)

ax.set_xlabel('Disponibilidad estimada en Colombia (%)',fontsize=11)
ax.set_ylabel('Caida en F1 si no disponible (pp)',fontsize=11)
ax.set_title('Criticidad de sensores para transferibilidad a Colombia\n'
             '(arriba-izquierda = sensor critico Y poco disponible = mayor riesgo)',
             fontsize=11,fontweight='bold')
plt.colorbar(sc,ax=ax,label='Caida F1 (pp)').ax.tick_params(labelsize=9)
ax.grid(linestyle='--',alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR/'criticidad_sensores_colombia.png',dpi=150,bbox_inches='tight')
plt.show(); print('Guardado: criticidad_sensores_colombia.png')

## 5. Resumen y conclusiones de transferibilidad

In [ ]:
print('='*65)
print('  RESUMEN — ABLACION Y ROBUSTEZ PARA TRANSFERIBILIDAD A COLOMBIA')
print('='*65)
print(f'\nLinea base RF (14 canales): F1 = {resultados["Linea base (14 ch)"][0]:.4f}')
print('\nImpacto de perder cada sensor:')
for k,v in list(resultados.items())[1:]:
    caida=(resultados['Linea base (14 ch)'][0]-v[0])*100
    riesgo='ALTO' if caida>2 else 'MEDIO' if caida>0.5 else 'BAJO'
    print(f'  {k:<30} caida={caida:+.2f} pp  riesgo={riesgo}')
print(f'\nRobustez ante nubosidad colombiana (60-80%):')
idx60=int(0.6*10); idx80=int(0.8*10)
print(f'  F1 con 60% nubes: {f1_cloud[6]:.4f}  (caida={( f1_cloud[0]-f1_cloud[6])*100:.2f} pp)')
print(f'  F1 con 80% nubes: {f1_cloud[8]:.4f}  (caida={(f1_cloud[0]-f1_cloud[8])*100:.2f} pp)')
print(f'  F1 solo SAR+DEM : {f1_sar_only[-1]:.4f} (escenario extremo sin optico)')
print(f'\nFiguras guardadas en: {OUT_DIR}')